Create a dataset of the numbers of job adverts in Wales and England for each occupation.

This is to label which occupations should be included in job quality analysis.

In [1]:
import polars as pl
import geopandas as gpd
import pandas as pd

from dap_prinz_green_jobs import PROJECT_DIR

import os

job_advert_s3_location = "s3://prinz-green-jobs/outputs/data/ojo_application"

In [3]:
titles_file = os.path.join(
    job_advert_s3_location,
    "deduplicated_sample/20241114/latest_update_20241114_titles.parquet",
)
titles_data = pl.read_parquet(titles_file)

In [4]:
combined_green_measures_filename = os.path.join(
    job_advert_s3_location,
    "extracted_green_measures/analysis/20241121/combined_green_measures_and_meta.parquet",
)

In [5]:
combined_all_data_orig = pl.read_parquet(combined_green_measures_filename)

In [6]:
len(combined_all_data_orig)

5967229

## Aggregate values

In [7]:
df = combined_all_data_orig[['job_id', 'PROP_GREEN', 'GREEN/NOT GREEN', 'GREEN TIMESHARE', 'INDUSTRY GHG PER UNIT EMISSIONS', 'itl_1_name']]

df = df.with_columns((pl.col("itl_1_name")=="Wales").alias("in_wales"))
df = df.join(titles_data[['id','parent_sector', 'sector']], left_on='job_id', right_on='id')

In [8]:
df.head(2)

job_id,PROP_GREEN,GREEN/NOT GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,itl_1_name,in_wales,parent_sector,sector
i64,f64,str,f64,f64,str,bool,str,str
41547517,0.0,null,null,0.0,"""London""",false,"""Financial Services""","""Other Financial Services"""
41547520,0.0,null,null,0.0,"""South East (England)""",false,"""Financial Services""","""Other Financial Services"""


In [9]:
df_pd = df.to_pandas()
df_pd['GREEN/NOT GREEN'] = df_pd['GREEN/NOT GREEN']=='Green'

In [10]:
t_alt_means = df_pd.groupby(['parent_sector', 'sector']).agg({
    'PROP_GREEN': 'mean',
    'GREEN TIMESHARE': 'mean',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    'GREEN/NOT GREEN': 'mean',
    
}).reset_index()

In [11]:
t_alt_means.head(2)

,parent_sector,sector,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN
0,Accountancy,Accounts Admin,0.001594,0.949072,0.083862,0.070922
1,Accountancy,Accounts Assistant,0.000729,0.121898,0.057470,0.018451


In [12]:
t_alt_wales = df_pd.groupby(['parent_sector', 'sector', 'in_wales']).agg({
    'job_id': 'count',
    
}).reset_index()
t_alt_wales.head(2)

all_counts_wales = t_alt_wales.pivot(index=['parent_sector',  'sector'], columns='in_wales')['job_id'].reset_index()
all_counts_wales.head(2)

in_wales,parent_sector,sector,False,True
0,Accountancy,Accounts Admin,25549.0,525.0
1,Accountancy,Accounts Assistant,56323.0,1254.0


In [13]:
print(len(t_alt_means))
print(len(all_counts_wales))

867
867


In [14]:
t_alt_means = t_alt_means.merge(all_counts_wales, on=['parent_sector',  'sector'])
len(t_alt_means)

867

In [15]:
t_alt_means.to_csv("job_ad_counts_per_sector.csv", index=False)

## Most green jobs using SOC EXT
- The problem with this is that the number of job adverts is so small for Wales so some highly green socs are too small

In [16]:
df_soc = combined_all_data_orig[['job_id', 'PROP_GREEN', 'GREEN/NOT GREEN',
                                 'GREEN TIMESHARE', 'INDUSTRY GHG PER UNIT EMISSIONS', 'itl_1_name', 'SOC_2020_EXT_name']]
df_soc = df_soc.with_columns((pl.col("itl_1_name")=="Wales").alias("in_wales"))
df_soc = df_soc.to_pandas()
df_soc['GREEN/NOT GREEN'] = df_soc['GREEN/NOT GREEN']=='Green'
df_soc_means = df_soc.groupby(['SOC_2020_EXT_name']).agg({
    'job_id': 'count',
    'PROP_GREEN': 'mean',
    'GREEN TIMESHARE': 'mean',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    'GREEN/NOT GREEN': 'mean',
}).reset_index()

In [17]:
df_soc_wales = df_soc.groupby(['SOC_2020_EXT_name', 'in_wales']).agg({
    'job_id': 'count',
}).reset_index()

df_soc_wales = df_soc_wales.pivot(index=['SOC_2020_EXT_name'], columns='in_wales')['job_id'].reset_index()
df_soc_wales.rename(columns={False: "Number not in Wales", True: "Number in Wales"}, inplace=True)
df_soc_wales.head(2)

in_wales,SOC_2020_EXT_name,Number not in Wales,Number in Wales
0,Accounting clerks and bookkeepers,194833.0,3697.0
1,Accounting technicians,1055.0,32.0


In [18]:
df_soc_means = df_soc_means.merge(df_soc_wales, on=['SOC_2020_EXT_name'])
df_soc_means.head(2)

,SOC_2020_EXT_name,job_id,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN,Number not in Wales,Number in Wales
0,Accounting clerks and bookkeepers,203034,0.000983,0.0,0.064961,0.0,194833.0,3697.0
1,Accounting technicians,1137,0.004258,0.0,0.040025,0.0,1055.0,32.0


In [19]:
top_sectors_1 = df_soc_means.sort_values(by="PROP_GREEN", ascending =False).head(20)['SOC_2020_EXT_name'].tolist()
top_sectors_2 = df_soc_means.sort_values(by="GREEN TIMESHARE", ascending = False).head(20)['SOC_2020_EXT_name'].tolist()
top_sectors_3 = df_soc_means.sort_values(by="INDUSTRY GHG PER UNIT EMISSIONS", ascending =True).head(20)['SOC_2020_EXT_name'].tolist()
top_sectors_4 = df_soc_means.sort_values(by="GREEN/NOT GREEN", ascending = False).head(20)['SOC_2020_EXT_name'].tolist()

In [20]:
print(top_sectors_1)

['Zookeepers', 'Environmental consultants ', 'Hydrogeologists and hydrologists', 'Solar panel installers', 'Panel beaters (excludes vehicles)', 'Sustainability officers', 'Environmental consultancy directors', 'Environmental and geo-environmental engineers', 'Ecologists', 'Energy managers', 'Recycling managers', 'Environmental professionals n.e.c.', 'Environmental health professionals', 'Waste management officers', 'Recycling operatives', 'Singers', 'Energy advisers and assessors', 'Sales energy consultants', 'Forest workers', 'Conservators']


In [21]:
print(top_sectors_2)

['Production managers and directors in manufacturing', 'Environmental and geo-environmental engineers', 'Sustainability officers', 'Energy managers', 'Environmental consultants ', 'Environmental professionals n.e.c.', 'Landfill site managers', 'Recycling managers', 'Scrap yard managers', 'Sewage works and water treatment managers', 'Conservation professionals n.e.c.', 'Conservationists ', 'Ecologists', 'Heritage officers', 'Managers and directors in the extraction of fossil fuels', 'Managers and directors in the production of energy', 'Production managers and directors in mining and energy n.e.c.', 'Waste disposal and environmental services managers n.e.c.', 'Road construction supervisors', 'Roofing supervisors']


In [22]:
print(top_sectors_3)

['Careers advisers and vocational guidance specialists n.e.c.', 'Divers', 'Parish clerks', 'Barristers and judges n.e.c.', 'Criminal solicitors and lawyers', 'Mortgage administrators', 'Stenographers', 'Family solicitors and lawyers', 'Mortgage advisers', 'Judges', 'Croupiers', 'Betting shop and gambling establishment managers n.e.c.', 'Chief superintendents', 'Casino managers ', 'Forensic accountants', 'Film and television runners', 'Clairvoyants, mediums and astrologers', 'Authors', 'Medical and scientific illustrators', 'Senior officers in immigration services']


In [23]:
print(top_sectors_4)

['Zoological scientists', 'Farmers n.e.c.', 'Falconers', 'Risk analysts', 'Road construction operatives', 'Road construction supervisors', 'Examinations officers', 'Road traffic and transport safety officers', 'Road traffic managers', 'Roadside assistance technicians', 'Estimators, valuers and assessors n.e.c.', 'Estimators ', 'Estimating managers and directors', 'Robotics engineers', 'Roofers, roof tilers and slaters n.e.c. ', 'Roofing supervisors', 'Routine inspectors and testers', 'Environmental scientists', 'Environmental professionals n.e.c.', 'Environmental consultants ']


In [24]:
# Taking everything from the above lists, but removing anything dodgy looking (e.g. health and safety)
bespoke_sector_list = [
    'Environmental consultants ', 'Hydrogeologists and hydrologists', 'Solar panel installers',
    'Sustainability officers', 'Environmental consultancy directors',
    'Environmental and geo-environmental engineers', 'Ecologists', 'Energy managers', 'Recycling managers',
    #'Environmental professionals n.e.c.', # removing since it could be confusing
    'Environmental health professionals', 'Waste management officers',
    'Recycling operatives', 'Energy advisers and assessors',
    'Forest workers', 'Conservators', #
    'Environmental consultants ', 'Energy managers',
    'Sustainability officers', 'Environmental and geo-environmental engineers',
    'Landfill site managers', 'Recycling managers', 'Scrap yard managers', 'Sewage works and water treatment managers',
    'Heritage officers', 'Ecologists', 'Conservationists ',
    # 'Waste disposal and environmental services managers n.e.c.',#removing since it could be confusing
    #
     'Environmental scientists',
    'Environmental consultants ',
]
print(len(bespoke_sector_list))

bespoke_sector_list_unique = set(bespoke_sector_list)
print(len(bespoke_sector_list_unique))

28
21


In [25]:
subset_soc_data = df_soc_means[df_soc_means['SOC_2020_EXT_name'].isin(bespoke_sector_list_unique)].sort_values(by='job_id', ascending=False
                                                                                            ).reset_index(drop=True)
subset_soc_data

,SOC_2020_EXT_name,job_id,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN,Number not in Wales,Number in Wales
0,Sustainability officers,3481,0.197434,62.50,0.437076,1.000000,3307.0,94.0
1,Ecologists,2581,0.157706,57.10,1.541626,1.000000,2331.0,108.0
2,Environmental consultants,1474,0.212676,62.50,0.263533,1.000000,1380.0,48.0
3,Environmental health professionals,1404,0.141128,8.50,0.750281,0.000000,1332.0,32.0
4,Environmental and geo-environmental engineers,1045,0.167495,62.50,0.194227,1.000000,949.0,28.0
5,Recycling operatives,1022,0.127884,40.40,0.983528,1.000000,942.0,34.0
6,Energy managers,716,0.150493,62.50,1.120294,1.000000,694.0,6.0
7,Energy advisers and assessors,704,0.124923,25.10,0.547140,1.000000,679.0,13.0
8,Conservationists,582,0.092121,57.10,0.920952,1.000000,554.0,17.0
9,Recycling managers,465,0.144952,58.40,1.128722,1.000000,427.0,30.0


## Aggregate by a bigger grouping

In [26]:
print(titles_data['parent_sector'].n_unique())
print(titles_data['sector'].n_unique())
print(titles_data['knowledge_domain'].n_unique())
print(titles_data['occupation'].n_unique())

38
794
15
6153


In [27]:
df_sector = combined_all_data_orig[['job_id', 'PROP_GREEN', 'GREEN/NOT GREEN',
                                 'GREEN TIMESHARE', 'INDUSTRY GHG PER UNIT EMISSIONS', 'itl_1_name']]
df_sector = df_sector.with_columns((pl.col("itl_1_name")=="Wales").alias("in_wales"))
df_sector = df_sector.join(titles_data[['id', 'sector']], left_on='job_id', right_on='id')
df_sector = df_sector.to_pandas()
df_sector['GREEN/NOT GREEN'] = df_sector['GREEN/NOT GREEN']=='Green'
df_sector_means = df_sector.groupby(['sector']).agg({
    'job_id': 'count',
    'PROP_GREEN': 'mean',
    'GREEN TIMESHARE': 'mean',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    'GREEN/NOT GREEN': 'mean',
}).reset_index()

In [28]:
len(df_sector_means)

793

In [29]:
df_sector_wales = df_sector.groupby(['sector', 'in_wales']).agg({
    'job_id': 'count',
}).reset_index()

df_sector_wales = df_sector_wales.pivot(index=['sector'], columns='in_wales')['job_id'].reset_index()
df_sector_wales.rename(columns={False: "Number not in Wales", True: "Number in Wales"}, inplace=True)

In [30]:
df_sector_means = df_sector_means.merge(df_sector_wales, on=['sector'])

In [31]:
top_sectors_1 = df_sector_means.sort_values(by="PROP_GREEN", ascending =False).head(20)['sector'].tolist()
top_sectors_2 = df_sector_means.sort_values(by="GREEN TIMESHARE", ascending = False).head(20)['sector'].tolist()
top_sectors_3 = df_sector_means.sort_values(by="INDUSTRY GHG PER UNIT EMISSIONS", ascending =True).head(20)['sector'].tolist()
top_sectors_4 = df_sector_means.sort_values(by="GREEN/NOT GREEN", ascending = False).head(20)['sector'].tolist()

In [32]:
print(top_sectors_1+top_sectors_2+top_sectors_3+top_sectors_4)

['Energy Advisor', 'Environmental Science', 'Ecology', 'Conservation/Environment', 'Renewable Energy', 'Pest Control', 'Environmental', 'Waste &amp; Recycling', 'Geoscience', 'Waste Management', 'Management / Operations', 'Geotechnical', 'Forestry', 'Earth Science', 'Other Energy', 'Water &amp; Environmental Consultancy', 'Health &amp; Safety Manager/SHEQ Advisor', 'Biomedical Science', 'Health &amp; Safety', 'Asbestos Surveyor', 'Ecology', 'Production Manager', 'Water &amp; Environmental Consultancy', 'Conservation/Environment', 'Technical Management', 'Environmental Science', 'Geotechnical', 'Environmental', 'Sales Director', 'Quantity Surveyor/PQS', 'Area Manager', 'Marketing Director', 'Workshop Manager', 'Town Planning', 'D&amp;B Co-ordinator', 'Energy Advisor', 'Warehouse Supervisor', 'Factory/Floor Manager', 'Warehouse Operative', 'Health &amp; Safety Manager/SHEQ Advisor', 'Development', 'Offshore Banking', 'Corp Actions/Divs', 'External Auditor (Non-Qual.)', 'External Auditor'

In [33]:
# Taking everything from the above lists, but removing anything dodgy looking (e.g. health and safety)
bespoke_sector_list = [
    'Energy Advisor', 'Environmental Science', 'Ecology', 'Conservation/Environment', 'Renewable Energy',
    'Pest Control', 'Environmental', 'Waste &amp; Recycling', 'Geoscience', 'Waste Management',
    'Geotechnical', 'Forestry', 'Earth Science', 
    'Water &amp; Environmental Consultancy',  'Ecology', 'Water &amp; Environmental Consultancy',
    'Conservation/Environment',  'Environmental Science', 'Geotechnical', 'Environmental', 
    'Energy Advisor',  'Animal Care', 'Welding/Plating/Pipefitting', 'Ecology','Geotechnical',
    'Thermal Engineer',
# 'Biomedical Science',
    'Other Energy',
]
print(len(bespoke_sector_list))

bespoke_sector_list_unique = set(bespoke_sector_list)
print(len(bespoke_sector_list_unique))

27
18


In [34]:
subset_sector_data = df_sector_means[df_sector_means['sector'].isin(bespoke_sector_list_unique)].sort_values(
    by='Number in Wales', ascending=False).reset_index(drop=True)
subset_sector_data

,sector,job_id,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN,Number not in Wales,Number in Wales
0,Environmental,10686,0.098345,28.459950,0.530669,0.722721,9911.0,370.0
1,Welding/Plating/Pipefitting,7307,0.009172,3.394620,0.319528,0.940468,6850.0,176.0
2,Conservation/Environment,3137,0.111350,31.327342,0.925239,0.639464,2893.0,130.0
3,Other Energy,5962,0.055974,18.370996,0.675573,0.479369,5615.0,124.0
4,Renewable Energy,7279,0.104360,17.415229,0.781177,0.513395,6721.0,123.0
5,Energy Advisor,2274,0.137500,25.994276,0.540337,0.550132,2135.0,85.0
6,Waste &amp; Recycling,1241,0.087586,21.298229,1.026085,0.628525,1123.0,73.0
7,Geotechnical,2948,0.064781,28.496148,0.139374,0.890095,2745.0,63.0
8,Waste Management,1112,0.075816,20.095253,0.808630,0.518885,1010.0,56.0
9,Environmental Science,952,0.130664,28.537594,0.657087,0.650210,850.0,54.0


In [35]:
subset_sector_data[subset_sector_data['Number in Wales']>10]

,sector,job_id,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN,Number not in Wales,Number in Wales
0,Environmental,10686,0.098345,28.459950,0.530669,0.722721,9911.0,370.0
1,Welding/Plating/Pipefitting,7307,0.009172,3.394620,0.319528,0.940468,6850.0,176.0
2,Conservation/Environment,3137,0.111350,31.327342,0.925239,0.639464,2893.0,130.0
3,Other Energy,5962,0.055974,18.370996,0.675573,0.479369,5615.0,124.0
4,Renewable Energy,7279,0.104360,17.415229,0.781177,0.513395,6721.0,123.0
5,Energy Advisor,2274,0.137500,25.994276,0.540337,0.550132,2135.0,85.0
6,Waste &amp; Recycling,1241,0.087586,21.298229,1.026085,0.628525,1123.0,73.0
7,Geotechnical,2948,0.064781,28.496148,0.139374,0.890095,2745.0,63.0
8,Waste Management,1112,0.075816,20.095253,0.808630,0.518885,1010.0,56.0
9,Environmental Science,952,0.130664,28.537594,0.657087,0.650210,850.0,54.0


In [36]:
main_sectors = subset_sector_data[subset_sector_data['Number in Wales']>10]['sector'].to_list()
print(main_sectors)

['Environmental', 'Welding/Plating/Pipefitting', 'Conservation/Environment', 'Other Energy', 'Renewable Energy', 'Energy Advisor', 'Waste &amp; Recycling', 'Geotechnical', 'Waste Management', 'Environmental Science', 'Water &amp; Environmental Consultancy', 'Ecology']


In [37]:
subset_sector_data[subset_sector_data['Number in Wales']>10]['job_id'].sum()

47551

## Subset and save out the job descriptions for these sectors only

In [4]:
main_sectors = ['Environmental', 'Welding/Plating/Pipefitting', 'Conservation/Environment', 'Other Energy',
                'Renewable Energy', 'Energy Advisor', 'Waste &amp; Recycling', 'Geotechnical', 'Waste Management',
                'Environmental Science', 'Water &amp; Environmental Consultancy', 'Ecology']

In [5]:
subset_titles = titles_data.filter(pl.col('sector').is_in(main_sectors))
len(subset_titles)

47551

In [7]:
subset_ids = set(subset_titles['id'].to_list())

In [6]:
job_descriptions = pl.read_parquet(
				"s3://prinz-green-jobs/outputs/data/ojo_application/deduplicated_sample/20241114/latest_update_20241114_descriptions.parquet"
			)

In [8]:
subset_job_descriptions = job_descriptions.filter(pl.col('id').is_in(subset_ids))

In [9]:
len(subset_job_descriptions)

47551

In [12]:
subset_job_descriptions.to_pandas().to_parquet(
    "s3://open-jobs-lake/job_quality/welsh_analysis/subset_green_sectors_descriptions.parquet")

2025-03-25 18:16:04,484 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


## Least green sectors

In [38]:
least_sectors_1 = df_sector_means.sort_values(by="PROP_GREEN", ascending =True).head(20)['sector'].tolist()
least_sectors_2 = df_sector_means.sort_values(by="GREEN TIMESHARE", ascending = True).head(20)['sector'].tolist()
least_sectors_3 = df_sector_means.sort_values(by="INDUSTRY GHG PER UNIT EMISSIONS", ascending =False).head(20)['sector'].tolist()
least_sectors_4 = df_sector_means.sort_values(by="GREEN/NOT GREEN", ascending = True).head(20)['sector'].tolist()

print(least_sectors_1+least_sectors_2+least_sectors_3+least_sectors_4)

['Team Leader Temporaries', 'Disability Inclusion', 'Development', 'Leisure &amp; Tourism', 'Resident Medical Officer', 'Partnership Secretary', 'Regeneration', 'Audiology', 'Retail Financial Advisor', 'Desktop Publishing', 'UAT', 'Arts &amp; Media', 'Valuer/Lister', 'Probate', 'Oncology Nurse', 'Employee Benefits', 'Vehicle Sales', 'Animation', 'Estate Agent', 'Billing Co-ordinator', 'Animal Care', 'Chiropody &amp; Podiatry', 'Pastry Chef', 'Speech &amp; Language Therapy', 'Chef de Partie', 'Sous-Chef', 'Paediatric Nurse', 'Commis-Chef', 'Purchase Ledger Clerk', 'Invigilator', 'Sales Ledger Clerk', 'Head/Executive Chef', 'Midwifery', 'Phlebotomy', 'Accounts Assistant', 'External Auditor (Non-Qual.)', 'Payroll', 'Bookkeeper', 'Key Stage 2', 'Community Psychiatric Nurse', 'Import Clerk', 'Mining &amp; Resources', 'Export Clerk', 'Cruise', 'Shipping', 'Import Co-Ordinator', 'Gardener', 'Airline', 'Ecology', 'Toxicology', 'Pest Control', 'Dietetics', 'Agriculture &amp; Horticulture', 'Agr

In [45]:
# Taking everything from the above lists, but removing anything dodgy looking
bespoke_not_green_sector_list = ['Team Leader Temporaries', 'Disability Inclusion', 'Development', 'Leisure &amp; Tourism',
 'Resident Medical Officer', 'Partnership Secretary', 'Regeneration', 'Audiology',
                                 'Retail Financial Advisor', 'Desktop Publishing', 'UAT',
                                 'Arts &amp; Media', 'Valuer/Lister', 'Probate', 'Oncology Nurse',
                                 'Employee Benefits', 'Vehicle Sales', 'Animation', 'Estate Agent',
                                 'Billing Co-ordinator', 'Chiropody &amp; Podiatry',
                                 'Speech &amp; Language Therapy',
                                 'Paediatric Nurse',
                                  'Purchase Ledger Clerk', 'Invigilator',
                                 'Sales Ledger Clerk',  'Midwifery',
                                 'Phlebotomy', 'Accounts Assistant', 'External Auditor (Non-Qual.)',
                                 'Payroll', 'Bookkeeper', 'Key Stage 2', 'Community Psychiatric Nurse',
                                 'Import Clerk', 'Mining &amp; Resources', 'Export Clerk',
                                 'Cruise', 'Shipping', 'Import Co-Ordinator',
                                 'Airline', 'Pest Control', 'Dietetics',
                                 'Agriculture &amp; Horticulture',
                                 'Agriculture', 'Transport Planner', 'Depot Manager',
                                 'Other Transport &amp; Logistics', 
                                 'Chiropody &amp; Podiatry', 'Speech &amp; Language Therapy',
                                 'Paediatric Nurse', 'Development', 
                                 'Nurse - Grade E', 'Key Stage 2',
                                  'Midwifery', 'ODAs/ODPs/Theatre Nurses',
                                 'Registered Mental Health Nurse', 'Nurse - Grade F',
                                  'Physiotherapy', 'Educational Psychology', 'Legal Cashier']

print(len(bespoke_not_green_sector_list))

bespoke_not_green_sector_list_unique = set(bespoke_not_green_sector_list)
print(len(bespoke_not_green_sector_list_unique))

61
55


In [48]:
subset_notgreen_sector_data = df_sector_means[df_sector_means['sector'].isin(bespoke_not_green_sector_list_unique)].sort_values(
    by='Number in Wales', ascending=False).reset_index(drop=True)

In [54]:
# Second pass, more than 20 welsh job ads but also a mix of the obviously not green + some with no green link
bespoke_not_green_sector_list_unique = [
   'Other Transport &amp; Logistics', 
    'Vehicle Sales',
     'Transport Planner',
    'Airline',
     'Shipping',
         'Export Clerk',
     'Depot Manager',
        'Accounts Assistant',
        'Payroll',
     'Estate Agent',
     'Registered Mental Health Nurse',
     'Key Stage 2',
]

df_sector_means[df_sector_means['sector'].isin(bespoke_not_green_sector_list_unique)]

,sector,job_id,PROP_GREEN,GREEN TIMESHARE,INDUSTRY GHG PER UNIT EMISSIONS,GREEN/NOT GREEN,Number not in Wales,Number in Wales
12,Accounts Assistant,58912,0.000729,0.121898,0.057470,0.018451,56323.0,1254.0
29,Airline,2580,0.011880,11.060559,1.760803,0.432171,2413.0,34.0
186,Depot Manager,2111,0.023921,18.992917,0.903557,0.711511,1993.0,57.0
231,Estate Agent,22248,0.000567,3.601318,0.039265,0.207030,20847.0,877.0
241,Export Clerk,4710,0.006065,3.452489,3.318335,0.492781,4552.0,61.0
372,Key Stage 2,21932,0.002470,0.137033,0.064450,0.002006,20859.0,318.0
528,Other Transport &amp; Logistics,33851,0.011070,11.223570,0.881457,0.435142,32031.0,805.0
545,Payroll,37803,0.001006,0.128735,0.069825,0.009629,36216.0,665.0
625,Registered Mental Health Nurse,10562,0.002429,0.221846,0.106007,0.002840,9697.0,480.0
683,Shipping,3521,0.007812,5.518740,2.397779,0.609202,3284.0,33.0


In [55]:
not_green_subset_titles = titles_data.filter(pl.col('sector').is_in(bespoke_not_green_sector_list_unique))
len(not_green_subset_titles)

214366

## Save combined info including salary for not green jobs

In [84]:
joined_combined_all_data_orig = combined_all_data_orig.join(titles_data[['id','parent_sector', 'sector']],
                                                            how='left', left_on='job_id', right_on='id')

In [86]:
notgreen_combined_data = joined_combined_all_data_orig.filter(pl.col('sector').is_in(bespoke_not_green_sector_list_unique)).to_pandas()
len(notgreen_combined_data)

214366

In [87]:
notgreen_combined_data["country"] = notgreen_combined_data['itl_1_name'].apply(
    lambda x: x if x in ["Wales", "Scotland", None] else "England")

In [113]:
notgreen_combined_data['mid_annualised_salary'] = notgreen_combined_data[['min_annualised_salary', 'max_annualised_salary']].mean(skipna=True,axis =1)

In [115]:
not_green_sectors_aggs = pd.DataFrame()

for sector_name in bespoke_not_green_sector_list_unique:
    sector_data = notgreen_combined_data[notgreen_combined_data['sector'] == sector_name]
    per_sector_aggs = sector_data.groupby("country", dropna=False).agg(
        {
            'job_id': 'count',
            'PROP_GREEN': 'mean',
            'GREEN TIMESHARE': 'mean',
            'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
            'min_annualised_salary': 'mean',
            'max_annualised_salary': 'mean',
            'mid_annualised_salary': 'mean',
        }).reset_index()
    per_sector_aggs["sector"] = sector_name.replace("&amp;", "&")
    not_green_sectors_aggs = pd.concat([not_green_sectors_aggs, per_sector_aggs])

In [117]:
not_green_sectors_aggs.rename(columns={
    "job_id": "Number of job adverts",
    'PROP_GREEN': "Average percentage of green skills",
    'GREEN TIMESHARE': "Average percentage of time spent on green tasks",
    'INDUSTRY GHG PER UNIT EMISSIONS': "Average GHG emissions",
    'min_annualised_salary': "Average minimum salary",
    'max_annualised_salary': "Average maximum salary",
    'mid_annualised_salary': "Average salary mid point",
}, inplace=True)

In [118]:
salary_melted_df = pd.melt(
    not_green_sectors_aggs.reset_index(drop=True).reset_index(),
    id_vars=['country', 'sector', 'index'], value_vars=["Average minimum salary", "Average maximum salary"]).round()
salary_melted_df = salary_melted_df[pd.notnull(salary_melted_df['country'])]
salary_melted_df.to_csv("welsh_salary_aggs_melt_not_green.csv", index=False)

In [120]:
salary_melted_df = pd.melt(
    not_green_sectors_aggs.reset_index(drop=True).reset_index(),
    id_vars=['country', 'sector', 'index'], value_vars=["Average salary mid point"]).round()
salary_melted_df = salary_melted_df[pd.notnull(salary_melted_df['country'])]
salary_melted_df.to_csv("welsh_mid_salary_aggs_melt_not_green.csv", index=False)